# Football Detect Current Parallel Experiments

Run order:
1. Confirm the repo root and install dependencies if needed.
2. Load `.env.staging` and confirm the staged Moondream keys.
3. Build the local football v2 split dataset if it is missing.
4. Inspect the split metadata before training.
5. Launch the current repaired football experiments in parallel.
6. Check status or wait for all runs to finish.

This notebook treats the current experiment set as the latest repaired `r32` football configs:
- `lr1e4_r32_g2_miou`
- `lr5e5_r32_g2_f1`
- `lr1e4_r32_g2_f1_promptfix`
- `lr1e4_r32_g2_f1_negmix`


In [13]:
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "football_detect" / "train_football_detect.py").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing football_detect/train_football_detect.py")


repo_root = find_repo_root(Path.cwd())
module_root = repo_root / "football_detect"

print("cwd:", Path.cwd().resolve())
print("repo_root:", repo_root)
print("module_root:", module_root)


cwd: /Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/football_detect
repo_root: /Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo
module_root: /Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/football_detect


In [14]:
# Install once if needed.
!python -m pip install -q datasets pillow numpy scipy wandb python-dotenv


## 1) Load `.env.staging`

The current parallel launch overrides API key env vars across the four runs so they do not all compete for `CICID_GPUB_MOONDREAM_API_KEY_4`.


In [15]:
import os
from dotenv import load_dotenv

loaded_files = []
for env_name, override in ((".env.staging", True), (".env", False)):
    env_path = repo_root / env_name
    if env_path.exists():
        load_dotenv(env_path, override=override)
        loaded_files.append(str(env_path.relative_to(repo_root)))

print("loaded env files:", loaded_files)
for key in [
    "CICID_GPUB_MOONDREAM_API_KEY_1",
    "CICID_GPUB_MOONDREAM_API_KEY_2",
    "CICID_GPUB_MOONDREAM_API_KEY_3",
    "CICID_GPUB_MOONDREAM_API_KEY_4",
    "HUGGINGFACE_HUB_TOKEN",
    "HF_TOKEN",
]:
    print(f"{key}:", "set" if os.getenv(key) else "MISSING")


loaded env files: ['.env.staging']
CICID_GPUB_MOONDREAM_API_KEY_1: set
CICID_GPUB_MOONDREAM_API_KEY_2: set
CICID_GPUB_MOONDREAM_API_KEY_3: set
CICID_GPUB_MOONDREAM_API_KEY_4: set
HUGGINGFACE_HUB_TOKEN: set
HF_TOKEN: MISSING


## 2) Build The Current Local V2 Splits If Needed

The current repaired `r32` configs use the local dataset path `football_detect/outputs/maxs-m87_football_detect_v2_splits`.


In [16]:
import subprocess
import sys

dataset_dir = repo_root / "football_detect" / "outputs" / "maxs-m87_football_detect_v2_splits"
generate_config = repo_root / "football_detect" / "configs" / "generate_football_splits_v2.json"

if dataset_dir.exists():
    print("dataset already exists:", dataset_dir)
else:
    cmd = [
        sys.executable,
        str(repo_root / "football_detect" / "generate_football_splits.py"),
        "--config",
        str(generate_config),
    ]
    print("running:", " ".join(cmd))
    subprocess.run(cmd, cwd=repo_root, check=True)


running: /Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/.venv/bin/python /Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/football_detect/generate_football_splits.py --config /Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/football_detect/configs/generate_football_splits_v2.json


Saving the dataset (1/1 shards): 100%|██████████| 37/37 [00:00<00:00, 5941.85 examples/s]


saved splits to /Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/football_detect/outputs/maxs-m87_football_detect_v2_splits (train=330, val=35, post_val=37)


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Map: 100%|██████████| 330/330 [00:00<00:00, 2527.52 examples/s]

Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 12.95ba/s]
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...0gn/T/tmpmba32c4t.parquet:   2%|▏         | 2.08MB /  133MB            

Processing Files (0 / 1)      :   2%|▏         | 2.08MB /  133MB, 10.4MB/s  

Processing Files (0 / 1)      :   2%|▏         | 3.24MB /  133MB, 8.12MB/s  

Processing Files (0 / 1)      :   3%|▎         | 3.90MB /  133MB, 6.50MB/s  
New Data Upload               :   1%|          |  536kB / 94.4MB,  893kB/s  

Processing Files (0 / 1)      :   5%|▍         | 6.51MB /  133MB, 8.16MB/s  
New Data Upload               :   3%|▎         | 2.68MB / 94.4MB, 3.35MB/s  

Processing Files (0 / 1)      :   8%|▊         | 1

pushed dataset to maxs-m87/football_detect_v2 (train, validation, test)


## 3) Inspect The Current Split Metadata

This gives a quick sanity check on split sizes and available classes before starting the parallel runs.


In [17]:
import json

metadata_path = dataset_dir / "metadata.json"
stats_path = dataset_dir / "stats.json"

if not metadata_path.exists() or not stats_path.exists():
    raise RuntimeError(f"Expected metadata and stats under {dataset_dir}")

metadata = json.loads(metadata_path.read_text())
stats = json.loads(stats_path.read_text())

print("dataset_dir:", dataset_dir)
print("source_dataset:", metadata.get("source_dataset"))
print("split_sizes:", stats.get("split_sizes"))
print("class_catalog:", metadata.get("class_catalog"))
print("class_counts:")
for split_name, counts in stats.get("class_counts", {}).items():
    print(" ", split_name, counts)


dataset_dir: /Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/football_detect/outputs/maxs-m87_football_detect_v2_splits
source_dataset: None
split_sizes: {'post_val': 37, 'train': 330, 'val': 35}
class_catalog: [{'class_name': 'area of focus', 'prompt': 'main action area'}, {'class_name': 'ball holder', 'prompt': 'ball carrier'}, {'class_name': 'defensive line', 'prompt': 'defensive line'}, {'class_name': 'offensive line', 'prompt': 'offensive line'}, {'class_name': 'offensive line / defensive line', 'prompt': 'offensive and defensive lines engaged after the snap'}, {'class_name': 'players on the field', 'prompt': 'players on the field'}, {'class_name': 'tackle', 'prompt': 'tackle'}]
class_counts:
  post_val {'area of focus': 13, 'ball holder': 7, 'defensive line': 2, 'offensive line': 2, 'offensive line / defensive line': 6, 'players on the field': 6, 'tackle': 1}
  train {'area of focus': 104, 'ball holder': 59, 'defensive line': 26, 'offensive line': 26, 'offensive line / d

## 4) Launch The Current Football Experiments In Parallel

These are the current repaired `r32` configs.

By config, all four runs currently point at `CICID_GPUB_MOONDREAM_API_KEY_4`. For actual parallel execution, this notebook overrides the launched process env vars to rotate across `CICID_GPUB_MOONDREAM_API_KEY_1..4`.

Each run writes its own log file under `football_detect/logs/current_parallel/`.


In [18]:
import subprocess
import sys

parallel_specs = [
    {
        "config_path": repo_root / "football_detect" / "configs" / "cicd" / "cicd_train_football_detect_repaired_offpolicy_lr1e4_r32_g2_miou.json",
        "api_key_env_var": "CICID_GPUB_MOONDREAM_API_KEY_1",
    },
    {
        "config_path": repo_root / "football_detect" / "configs" / "cicd" / "cicd_train_football_detect_repaired_offpolicy_lr5e5_r32_g2_f1.json",
        "api_key_env_var": "CICID_GPUB_MOONDREAM_API_KEY_2",
    },
    {
        "config_path": repo_root / "football_detect" / "configs" / "cicd" / "cicd_train_football_detect_repaired_offpolicy_lr1e4_r32_g2_f1_promptfix.json",
        "api_key_env_var": "CICID_GPUB_MOONDREAM_API_KEY_3",
    },
    {
        "config_path": repo_root / "football_detect" / "configs" / "cicd" / "cicd_train_football_detect_repaired_offpolicy_lr1e4_r32_g2_f1_negmix.json",
        "api_key_env_var": "CICID_GPUB_MOONDREAM_API_KEY_4",
    },
]

required_keys = sorted({spec["api_key_env_var"] for spec in parallel_specs})
missing_keys = [key for key in required_keys if not os.getenv(key)]
if missing_keys:
    raise RuntimeError(f"Missing required API key env vars for parallel launch: {missing_keys}")

logs_dir = repo_root / "football_detect" / "logs" / "current_parallel"
logs_dir.mkdir(parents=True, exist_ok=True)

parallel_train_runs = []
for spec in parallel_specs:
    config_path = spec["config_path"]
    config_name = config_path.stem
    log_path = logs_dir / f"{config_name}.log"
    log_handle = log_path.open("w", encoding="utf-8")
    cmd = [
        sys.executable,
        str(repo_root / "football_detect" / "train_football_detect.py"),
        "--config",
        str(config_path),
        "--api-key-env-var",
        spec["api_key_env_var"],
    ]
    proc = subprocess.Popen(cmd, cwd=repo_root, stdout=log_handle, stderr=subprocess.STDOUT)
    parallel_train_runs.append(
        {
            "name": config_name,
            "config_path": str(config_path.relative_to(repo_root)),
            "api_key_env_var": spec["api_key_env_var"],
            "pid": proc.pid,
            "process": proc,
            "log_path": str(log_path),
            "log_handle": log_handle,
        }
    )
    print(
        f"started {config_name} | pid={proc.pid} | key={spec['api_key_env_var']} | log={log_path}"
    )

print(f"launched {len(parallel_train_runs)} parallel football runs")


started cicd_train_football_detect_repaired_offpolicy_lr1e4_r32_g2_miou | pid=31592 | key=CICID_GPUB_MOONDREAM_API_KEY_1 | log=/Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/football_detect/logs/current_parallel/cicd_train_football_detect_repaired_offpolicy_lr1e4_r32_g2_miou.log
started cicd_train_football_detect_repaired_offpolicy_lr5e5_r32_g2_f1 | pid=31593 | key=CICID_GPUB_MOONDREAM_API_KEY_2 | log=/Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/football_detect/logs/current_parallel/cicd_train_football_detect_repaired_offpolicy_lr5e5_r32_g2_f1.log
started cicd_train_football_detect_repaired_offpolicy_lr1e4_r32_g2_f1_promptfix | pid=31594 | key=CICID_GPUB_MOONDREAM_API_KEY_3 | log=/Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/football_detect/logs/current_parallel/cicd_train_football_detect_repaired_offpolicy_lr1e4_r32_g2_f1_promptfix.log
started cicd_train_football_detect_repaired_offpolicy_lr1e4_r32_g2_f1_negmix | pid=31595 | key=CICID_GPUB_MOONDREAM_AP

## 5) Check Parallel Run Status And Tail Recent Logs

Run this while the jobs are active to see process status and the last few log lines from each run.


In [19]:
if "parallel_train_runs" not in globals() or not parallel_train_runs:
    raise RuntimeError("Launch the parallel runs first.")

for run in parallel_train_runs:
    proc = run["process"]
    return_code = proc.poll()
    status = "running" if return_code is None else f"finished rc={return_code}"
    print(f"\n[{run['name']}] pid={run['pid']} key={run['api_key_env_var']} status={status}")
    log_path = Path(run["log_path"])
    if log_path.exists():
        lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
        for line in lines[-12:]:
            print(line)



[cicd_train_football_detect_repaired_offpolicy_lr1e4_r32_g2_miou] pid=31592 key=CICID_GPUB_MOONDREAM_API_KEY_1 status=running

[cicd_train_football_detect_repaired_offpolicy_lr5e5_r32_g2_f1] pid=31593 key=CICID_GPUB_MOONDREAM_API_KEY_2 status=running

[cicd_train_football_detect_repaired_offpolicy_lr1e4_r32_g2_f1_promptfix] pid=31594 key=CICID_GPUB_MOONDREAM_API_KEY_3 status=running

[cicd_train_football_detect_repaired_offpolicy_lr1e4_r32_g2_f1_negmix] pid=31595 key=CICID_GPUB_MOONDREAM_API_KEY_4 status=running


## 6) Wait For All Parallel Runs To Finish

This blocks until every launched process exits, then closes the log handles.


In [20]:
if "parallel_train_runs" not in globals() or not parallel_train_runs:
    raise RuntimeError("Launch the parallel runs first.")

for run in parallel_train_runs:
    rc = run["process"].wait()
    run["log_handle"].close()
    print(f"finished {run['name']} with rc={rc}")


finished cicd_train_football_detect_repaired_offpolicy_lr1e4_r32_g2_miou with rc=1
finished cicd_train_football_detect_repaired_offpolicy_lr5e5_r32_g2_f1 with rc=1
finished cicd_train_football_detect_repaired_offpolicy_lr1e4_r32_g2_f1_promptfix with rc=1
finished cicd_train_football_detect_repaired_offpolicy_lr1e4_r32_g2_f1_negmix with rc=1
